# 8. Shot execution at pace


### Purpose

Assess whether faster puck movement before a shot helps the attempt reach the net. Distinguish blocked attempts, missed attempts, and attempts on net, including goals. The coaching question is: **can the team turn faster puck movement into attempts that test the goalie?**

### Expected outcome

A shot-level research table and pooled slower/middle/faster pace comparisons, with shot-distance context, team summaries, and attempts for review. Connect this study with notebook 07's entry and shot-distance findings. Associations identify execution patterns to investigate, not proof that increasing pace causes better shooting outcomes.


## 8.1. Setup and load processed data

Use the same augmented events, reference tables, and transition pace as notebook 07. Include 5v5 only and pool the sources for simplicity. Preserve source provenance in the underlying shot records, while using shared pace boundaries in the analysis.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values("event_id")
transitions = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
teams = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
players = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {"events": events, "transitions": transitions,
                        "teams": teams, "games": games, "players": players}.items()
]))
print("Timing:", pace_manifest["timing"])


,table,rows,columns
0,events,86670,27
1,transitions,63298,25
2,teams,20,3
3,games,34,5
4,players,435,4


Timing: Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.


## 8.2. Define shot attempts, outcomes, and distance

Use `Shot` and `Goal` records once each, excluding penalty-shot goals. Goals count as on-net attempts. Shot outcomes come from `detail_2`: blocked, missed, or on net. A blocked attempt does not directly measure defensive structure, and a missed attempt does not by itself establish poor execution.

Calculate distance from the recorded attempt location to the attacking net center `(189, 42.5)` in normalized rink feet. Retain missing/invalid distance as unknown. Use broad distance bands: 0-20 feet, 20-40 feet, and over 40 feet. These comparisons help assess location differences, but do not adjust for shooting angle or defensive pressure.

Separate deflections from the primary analysis because their outcomes reflect a different action. Retain shot type, one-timer, traffic, team, and score context for review. Goal conversion is a secondary descriptive measure, not the primary endpoint.


In [2]:
shots = events.loc[events["event"].isin(["Shot", "Goal"]) & events["is_5v5"]].copy()
shots["shot_outcome"] = shots["detail_2"].replace({"On Net": "On net"})
shots.loc[shots["event"].eq("Goal"), "shot_outcome"] = "On net"
assert shots["shot_outcome"].isin(["Blocked", "Missed", "On net"]).all()
shots["on_net"] = shots["shot_outcome"].eq("On net")
shots["goal"] = shots["event"].eq("Goal")
shots["shot_type"] = shots["detail_1"].fillna("Unknown")
shots["is_deflection"] = shots["shot_type"].eq("Deflection")
shots["team_name"] = shots["team_id"].map(teams.set_index("team_id")["team_name"])
shots["player_name"] = shots["player_id"].map(players.set_index("player_id")["player_name"])
shots["game_date"] = shots["game_id"].map(games.set_index("game_id")["game_date"])
valid_location = (shots["x"].between(0, 200) & shots["y"].between(0, 85)).fillna(False)
shots["distance_to_net_ft"] = np.hypot(shots["x"] - 189, shots["y"] - 42.5).where(valid_location)
shots["distance_band"] = pd.cut(
    shots["distance_to_net_ft"].astype(float), bins=[0, 20, 40, np.inf],
    labels=["0-20 ft", "20-40 ft", "Over 40 ft"], include_lowest=True,
).astype("string").fillna("Unknown distance")
display(shots.groupby(["is_deflection", "shot_outcome"]).size().rename("attempts").reset_index())
print("Unknown distance:", shots["distance_to_net_ft"].isna().sum())


,is_deflection,shot_outcome,attempts
0,False,Blocked,881
1,False,Missed,743
2,False,On net,1403
3,True,Blocked,2
4,True,Missed,53
5,True,On net,62


Unknown distance: 0


## 8.3. Calculate possession-sequence pace before each attempt

Reuse the existing inferred possession IDs. Pool eligible movement distance and modeled duration from transitions ending before the focal shot event. Exclude movement ending at the shot and all subsequent movement; full exported sequence pace would include the outcome and is not used directly.

There is no fixed lookback. Unknown or zero-duration preceding pace remains missing and is excluded from pace bands, with counts reported. Earlier same-time events are ordered by event ID. Modeled durations are estimated rather than measured pass-flight or skating times.


In [3]:
valid_movements = transitions.loc[transitions["exclusion_reason"].isna()].copy()
# Reuse existing possession sequences; trim each one at the shot event.
movement_groups = {
    pid: group[["end_event_id", "distance_ft", "modeled_elapsed_seconds"]].to_numpy(dtype=float)
    for pid, group in valid_movements.groupby("possession_id", sort=False)
}
sequence_start = events.groupby("possession_id")["modeled_clock_seconds"].max()
pace_rows = []
for shot in shots.itertuples(index=False):
    movements = movement_groups.get(shot.possession_id)
    if movements is not None:
        movements = movements[movements[:, 0] < shot.event_id]
    count = 0 if movements is None else len(movements)
    distance = 0.0 if movements is None else float(movements[:, 1].sum())
    duration = 0.0 if movements is None else float(movements[:, 2].sum())
    pace_rows.append({
        "event_id": shot.event_id,
        "preceding_transitions": count,
        "preceding_distance_ft": distance,
        "preceding_movement_seconds": duration,
        "preceding_pace_ft_s": distance / duration if duration > 0 else np.nan,
        "available_history_seconds": float(
            sequence_start.loc[shot.possession_id] - shot.modeled_clock_seconds
        ),
        "pace_status": "eligible" if duration > 0 else "no_history" if count == 0 else "zero_duration",
    })
shots = shots.merge(pd.DataFrame(pace_rows), on="event_id", validate="one_to_one")
coverage = shots.groupby("pace_status").agg(
    shots=("event_id", "size"), outcome_rate=("on_net", "mean")
).reset_index()
display(coverage)


,pace_status,shots,outcome_rate
0,eligible,2739,0.47353
1,no_history,395,0.41519
2,zero_duration,10,0.4


## 8.4. Define pooled pace tertiles

Use non-deflected attempts with measurable preceding pace to define three equal-frequency bands: slower, middle, faster. Boundaries are shared across teams and distance bands. Keep ties together; duplicate quantile edges may reduce the number of bands.

Assign deflections to the same boundaries for a separate descriptive table; out-of-range values stay unassigned. Pooled patterns may reflect differences between competitions and recording practices. No source adjustment is made here.


In [4]:
analysis_shots = shots.loc[shots["pace_status"].eq("eligible") & ~shots["is_deflection"]].copy()
bins, pace_edges = pd.qcut(analysis_shots["preceding_pace_ft_s"], q=3,
                          labels=False, retbins=True, duplicates="drop")
band_names = {1: "Slower", 2: "Middle", 3: "Faster"}
analysis_shots["pace_tertile"] = (bins + 1).astype("Int64")
analysis_shots["pace_band"] = analysis_shots["pace_tertile"].map(band_names)
pace_boundaries = pd.DataFrame({
    "pace_tertile": np.arange(1, len(pace_edges)),
    "lower_pace_ft_s": pace_edges[:-1], "upper_pace_ft_s": pace_edges[1:],
})
pace_boundaries["pace_band"] = pace_boundaries["pace_tertile"].map(band_names)
display(pace_boundaries.round(2))
print(f"Primary analysis: {len(analysis_shots)} of {len(shots)} 5v5 attempts")


,pace_tertile,lower_pace_ft_s,upper_pace_ft_s,pace_band
0,1,0.00,17.94,Slower
1,2,17.94,23.62,Middle
2,3,23.62,160.42,Faster


Primary analysis: 2631 of 3144 5v5 attempts


## 8.5. Compare blocked, missed, and on-net attempts

Show outcome percentages as stacked bars without uncertainty intervals. Include attempt and game counts in the table and hover. Goals are included in on-net outcomes; goal percentage uses all attempts as its denominator.

A lower blocked percentage at higher pace is consistent with better lane access, but does not prove that pace opened the lane. A higher miss percentage may reflect shot difficulty or selection. The on-net percentage is the primary execution measure.


In [5]:
OUTCOME_ORDER = ["Blocked", "Missed", "On net"]
PACE_ORDER = ["Slower", "Middle", "Faster"]
OUTCOME_COLORS = {"Blocked": "#64748b", "Missed": "#ea580c", "On net": "#2563eb"}

def summarize_outcomes(data, keys):
    counts = data.groupby(keys + ["shot_outcome"], observed=True).size().unstack("shot_outcome", fill_value=0)
    counts = counts.reindex(columns=OUTCOME_ORDER, fill_value=0)
    counts["attempts"] = counts[OUTCOME_ORDER].sum(axis=1)
    summary = data.groupby(keys, observed=True).agg(
        games=("game_id", "nunique"), mean_distance_ft=("distance_to_net_ft", "mean"),
        median_pace_ft_s=("preceding_pace_ft_s", "median"), goals=("goal", "sum"),
    ).join(counts)
    for outcome in OUTCOME_ORDER:
        summary[f"{outcome}_pct"] = 100 * summary[outcome] / summary["attempts"]
    summary["goal_pct"] = 100 * summary["goals"] / summary["attempts"]
    return summary.reset_index()

outcome_summary = summarize_outcomes(analysis_shots, ["pace_band"])
display(outcome_summary.round(2))
outcome_plot = outcome_summary.melt(
    id_vars=["pace_band", "attempts", "games"],
    value_vars=[f"{outcome}_pct" for outcome in OUTCOME_ORDER],
    var_name="shot_outcome", value_name="outcome_pct",
)
outcome_plot["shot_outcome"] = outcome_plot["shot_outcome"].str.removesuffix("_pct")
outcome_fig = px.bar(
    outcome_plot, x="pace_band", y="outcome_pct", color="shot_outcome",
    category_orders={"pace_band": PACE_ORDER, "shot_outcome": OUTCOME_ORDER},
    color_discrete_map=OUTCOME_COLORS, hover_data=["attempts", "games"],
    labels={"pace_band": "Preceding sequence pace", "outcome_pct": "Attempts (%)", "shot_outcome": "Outcome"},
    title="5v5 shot execution at pace | non-deflected attempts",
)
outcome_fig.update_layout(template="plotly_white", height=450)
outcome_fig.update_yaxes(range=[0, 100])
outcome_fig.show()


,pace_band,games,mean_distance_ft,median_pace_ft_s,goals,Blocked,Missed,On net,attempts,Blocked_pct,Missed_pct,On net_pct,goal_pct
0,Faster,34,37.69,28.74,30,242,220,415,877,27.59,25.09,47.32,3.42
1,Middle,34,35.46,20.66,34,244,211,422,877,27.82,24.06,48.12,3.88
2,Slower,34,33.86,14.35,28,274,202,401,877,31.24,23.03,45.72,3.19


## 8.6. Compare within shot-distance bands

Repeat the outcome comparison within broad distance bands using the same pooled pace boundaries. Include an unknown-distance group if needed. Retain counts; a sparse distance/pace cell cannot establish a stable execution pattern.

These are descriptive splits rather than a fully adjusted model. Shot type, angle, traffic, one-timers, teammates, and opponents can still explain differences. Distance is measured at the recorded attempt location, which may have source-specific tagging differences.


In [6]:
distance_summary = summarize_outcomes(analysis_shots, ["distance_band", "pace_band"])
display(distance_summary.round(2))
distance_plot = distance_summary.melt(
    id_vars=["distance_band", "pace_band", "attempts", "games"],
    value_vars=[f"{outcome}_pct" for outcome in OUTCOME_ORDER],
    var_name="shot_outcome", value_name="outcome_pct",
)
distance_plot["shot_outcome"] = distance_plot["shot_outcome"].str.removesuffix("_pct")
distance_fig = px.bar(
    distance_plot, x="pace_band", y="outcome_pct", color="shot_outcome", facet_col="distance_band",
    category_orders={"pace_band": PACE_ORDER, "shot_outcome": OUTCOME_ORDER,
                     "distance_band": ["0-20 ft", "20-40 ft", "Over 40 ft", "Unknown distance"]},
    color_discrete_map=OUTCOME_COLORS, hover_data=["attempts", "games"],
    labels={"pace_band": "Preceding sequence pace", "outcome_pct": "Attempts (%)", "shot_outcome": "Outcome"},
    title="Shot execution at comparable distances | 5v5, non-deflected",
)
distance_fig.update_layout(template="plotly_white", height=450)
distance_fig.update_yaxes(range=[0, 100])
distance_fig.show()


,distance_band,pace_band,games,mean_distance_ft,median_pace_ft_s,goals,Blocked,Missed,On net,attempts,Blocked_pct,Missed_pct,On net_pct,goal_pct
0,0-20 ft,Faster,34,11.88,27.58,17,18,38,129,185,9.73,20.54,69.73,9.19
1,0-20 ft,Middle,34,11.45,20.83,22,20,47,148,215,9.30,21.86,68.84,10.23
2,0-20 ft,Slower,34,10.58,14.48,17,28,44,147,219,12.79,20.09,67.12,7.76
3,20-40 ft,Faster,34,30.53,28.74,9,75,63,156,294,25.51,21.43,53.06,3.06
4,20-40 ft,Middle,34,30.59,20.62,9,88,60,157,305,28.85,19.67,51.48,2.95
5,20-40 ft,Slower,34,30.33,14.08,9,106,77,142,325,32.62,23.69,43.69,2.77
6,Over 40 ft,Faster,34,54.97,29.45,4,149,119,130,398,37.44,29.90,32.66,1.01
7,Over 40 ft,Middle,34,54.09,20.61,3,136,104,117,357,38.10,29.13,32.77,0.84
8,Over 40 ft,Slower,34,52.61,14.53,2,140,81,112,333,42.04,24.32,33.63,0.6


## 8.7. Review teams, deflections, and individual attempts

Retain team profiles and shot-type context for coaching review. Show deflections separately with the primary analysis's pace boundaries. Unknown preceding pace stays outside these comparisons.

Before recommending a change, inspect actual sequences and check consistency across games and recorded-time sensitivity. The defensible finding is a context-specific pattern to review, not a blanket instruction to shoot faster or slow down.


In [7]:
team_summary = summarize_outcomes(analysis_shots, ["team_id", "team_name", "pace_band"])
shot_type_summary = summarize_outcomes(analysis_shots, ["shot_type", "pace_band"])
display(team_summary.round(2))
display(shot_type_summary.round(2))

deflections = shots.loc[shots["is_deflection"] & shots["pace_status"].eq("eligible")].copy()
deflections["pace_tertile"] = (pd.cut(
    deflections["preceding_pace_ft_s"], bins=pace_edges, labels=False, include_lowest=True,
) + 1).astype("Int64")
deflections["pace_band"] = deflections["pace_tertile"].map(band_names)
print("Deflections outside primary pace boundaries:", deflections["pace_band"].isna().sum())
if not deflections.dropna(subset=["pace_band"]).empty:
    display(summarize_outcomes(deflections.dropna(subset=["pace_band"]), ["pace_band"]).round(2))

REVIEW_PACE_BAND = "Faster"
REVIEW_OUTCOME = "Missed"
review_shots = analysis_shots.loc[
    analysis_shots["pace_band"].eq(REVIEW_PACE_BAND)
    & analysis_shots["shot_outcome"].eq(REVIEW_OUTCOME)
]
display(review_shots[[
    "game_id", "game_date", "team_name", "player_name", "period", "clock_seconds",
    "event_id", "possession_id", "preceding_pace_ft_s", "preceding_movement_seconds",
    "shot_outcome", "shot_type", "distance_to_net_ft", "detail_3", "detail_4",
]].head(20))


,team_id,team_name,pace_band,games,mean_distance_ft,median_pace_ft_s,goals,Blocked,Missed,On net,attempts,Blocked_pct,Missed_pct,On net_pct,goal_pct
0,1,Boston Pride,Faster,7,34.63,28.12,5,27,35,53,115,23.48,30.43,46.09,4.35
1,1,Boston Pride,Middle,7,32.38,20.77,4,23,25,41,89,25.84,28.09,46.07,4.49
2,1,Boston Pride,Slower,7,36.67,14.52,4,36,22,42,100,36.00,22.00,42.00,4.0
3,2,Buffalo Beauts,Faster,6,35.61,28.03,4,17,14,28,59,28.81,23.73,47.46,6.78
4,2,Buffalo Beauts,Middle,6,34.51,20.71,1,19,16,38,73,26.03,21.92,52.05,1.37
5,2,Buffalo Beauts,Slower,6,34.77,14.72,0,21,19,33,73,28.77,26.03,45.21,0.0
6,3,Connecticut Whale,Faster,4,39.79,27.85,4,14,10,30,54,25.93,18.52,55.56,7.41
7,3,Connecticut Whale,Middle,4,33.45,20.82,1,16,11,31,58,27.59,18.97,53.45,1.72
8,3,Connecticut Whale,Slower,4,33.66,14.24,1,23,12,33,68,33.82,17.65,48.53,1.47
9,4,Metropolitan Riveters,Faster,3,35.6,27.73,0,13,3,16,32,40.62,9.38,50.00,0.0


,shot_type,pace_band,games,mean_distance_ft,median_pace_ft_s,goals,Blocked,Missed,On net,attempts,Blocked_pct,Missed_pct,On net_pct,goal_pct
0,Fan,Faster,15,34.29,28.20,0,2,12,2,16,12.50,75.00,12.50,0.0
1,Fan,Middle,17,27.31,21.36,0,6,13,5,24,25.00,54.17,20.83,0.0
2,Fan,Slower,20,26.46,14.39,0,3,30,5,38,7.89,78.95,13.16,0.0
3,Slapshot,Faster,31,52.75,30.37,2,32,20,31,83,38.55,24.10,37.35,2.41
4,Slapshot,Middle,26,52.24,20.81,1,32,26,21,79,40.51,32.91,26.58,1.27
5,Slapshot,Slower,31,51.53,14.50,2,41,18,23,82,50.00,21.95,28.05,2.44
6,Snapshot,Faster,34,33.28,28.40,20,91,89,194,374,24.33,23.80,51.87,5.35
7,Snapshot,Middle,34,31.7,20.70,22,100,88,194,382,26.18,23.04,50.79,5.76
8,Snapshot,Slower,34,29.81,14.33,20,105,78,194,377,27.85,20.69,51.46,5.31
9,Wrap Around,Faster,8,4.67,26.56,0,2,4,4,10,20.00,40.00,40.00,0.0


Deflections outside primary pace boundaries: 0


,pace_band,games,mean_distance_ft,median_pace_ft_s,goals,Blocked,Missed,On net,attempts,Blocked_pct,Missed_pct,On net_pct,goal_pct
0,Faster,21,14.18,25.51,5,1,16,20,37,2.70,43.24,54.05,13.51
1,Middle,22,15.87,20.48,3,1,17,20,38,2.63,44.74,52.63,7.89
2,Slower,21,15.59,15.01,5,0,14,19,33,0.00,42.42,57.58,15.15


,game_id,game_date,team_name,player_name,period,clock_seconds,event_id,possession_id,preceding_pace_ft_s,preceding_movement_seconds,shot_outcome,shot_type,distance_to_net_ft,detail_3,detail_4
2,3,2021-01-23,Minnesota Whitecaps,Nina Rodgers,1,1147.0,39,9,35.495748,4.500000,Missed,Snapshot,32.927952,No traffic,Not a one timer
12,3,2021-01-23,Minnesota Whitecaps,Lynn Astrup,1,904.0,209,39,36.833099,4.000000,Missed,Snapshot,31.705678,Traffic,Not a one timer
31,3,2021-01-23,Minnesota Whitecaps,Allie Thunstrom,2,1121.0,793,165,28.534851,2.666667,Missed,Wristshot,25.831183,Traffic,Not a one timer
32,3,2021-01-23,Minnesota Whitecaps,Emma Stauber,2,1118.0,795,165,26.764614,7.000000,Missed,Wristshot,64.507751,Traffic,Not a one timer
75,3,2021-01-23,Minnesota Whitecaps,Maddie Rowe,3,763.0,1770,330,23.672140,11.500000,Missed,Slapshot,70.001786,Traffic,One timer
101,1,2021-01-23,Connecticut Whale,Hanna Beattie,1,651.0,2698,506,25.283062,15.500000,Missed,Snapshot,69.211632,No traffic,One timer
153,1,2021-01-23,Buffalo Beauts,Brittany Colton,2,95.0,3937,782,26.218060,5.000000,Missed,Snapshot,45.620719,Traffic,Not a one timer
154,1,2021-01-23,Buffalo Beauts,Whitney Dove,2,88.0,3941,782,25.263477,12.666667,Missed,Snapshot,49.449469,Traffic,Not a one timer
197,1,2021-01-23,Buffalo Beauts,Kristin Lewicki,3,181.0,4711,923,45.021917,0.666667,Missed,Snapshot,12.5,No traffic,Not a one timer
265,2,2021-01-23,Toronto Six,Lindsay Eastwood,3,1136.0,6665,1314,60.464866,0.500000,Missed,Slapshot,59.22204,Traffic,Not a one timer
